# 第 8 章: Survived データの探索と可視化

クラスの偏り、性別・客室クラス別の生存率、木の深さとクラスの重みの効果、混同行列、分割に使われた特徴量を確認する。

In [ ]:
%use dataframe(0.15.0), kandy(0.8.0)
@file:DependsOn("org.tribuo:tribuo-classification-tree:4.3.2")

In [ ]:
@file:DependsOn("../build/libs/getting-started-ml.jar")

In [ ]:
import chapter02.splitTrainTest
import chapter03.trainTribuoTree
import chapter08.ClassWeight
import chapter08.buildPipeline
import chapter08.evaluate
import chapter08.loadSurvived
import chapter08.splitFeaturesAndTarget
import org.tribuo.classification.Label
import org.tribuo.common.tree.TreeModel
import java.io.File

// Notebook は notebooks/ で実行されるので、学習データの既定の場所を 1 つ上にずらす
val survivedCsv = File(dataset.dataDir { name -> System.getenv(name) ?: "../../data/sukkiri-ml" }, "Survived.csv")
val df = loadSurvived(survivedCsv)
val (x, t) = splitFeaturesAndTarget(df)
val split = splitTrainTest(x, t, testSize = 0.2, seed = 0)

## クラス分布

In [ ]:
val classCounts = df.groupBy("Survived").count()
classCounts.plot {
    bars {
        x("Survived")
        y("count")
    }
    layout.title = "生存（1）と死亡（0）の人数"
}

In [ ]:
classCounts

## 性別・客室クラス別の生存率

`Survived` は 0 と 1 なので、平均値が生存率になる。

In [ ]:
val survivalRate = df.groupBy("Pclass", "Sex").mean("Survived").sortBy("Pclass", "Sex")
survivalRate.plot {
    bars {
        x("Pclass")
        y("Survived")
        fillColor("Sex")
    }
    layout.title = "客室クラス・性別ごとの生存率"
}

In [ ]:
survivalRate

## 木の深さとクラスの重み

`maxDepth` を 1 から 10 まで変え、クラスの重みの有無で正解率と見つけた生存者の数を比べる。

In [ ]:
val scores =
    ClassWeight.entries
        .flatMap { classWeight ->
            (1..10).map { depth ->
                val evaluation = evaluate(buildPipeline(depth, classWeight).fit(split.xTrain, split.tTrain), split)
                listOf(classWeight.name, depth, evaluation.trainAccuracy, evaluation.testAccuracy, evaluation.foundSurvivors)
            }
        }.let { rows ->
            dataFrameOf("classWeight", "depth", "train", "test", "foundSurvivors")(*rows.flatten().toTypedArray())
        }
scores.plot {
    line {
        x("depth")
        y("test")
        color("classWeight")
    }
    points {
        x("depth")
        y("test")
        color("classWeight")
    }
    layout.title = "木の深さとテストデータの正解率"
}

In [ ]:
scores

## 混同行列

深さ 5・`BALANCED` のパイプラインで、テストデータの予測と実際を突き合わせる。

In [ ]:
val balanced = buildPipeline(maxDepth = 5, classWeight = ClassWeight.BALANCED).fit(split.xTrain, split.tTrain)
val pairs = balanced.predict(split.xTest).zip(split.tTest)
dataFrameOf("実際", "死亡と予測", "生存と予測")(
    "死亡", pairs.count { it == 0 to 0 }, pairs.count { it == 1 to 0 },
    "生存", pairs.count { it == 0 to 1 }, pairs.count { it == 1 to 1 },
)

## 分割に使われた特徴量

前処理した訓練データで Tribuo の決定木（深さ 5、重み付けなし）を学習し、`getTopFeatures` で特徴量が分割に使われた回数を見る（第 3 章で確かめたとおり、不純度の減少量ではなく回数）。

In [ ]:
val none = buildPipeline(maxDepth = 5, classWeight = ClassWeight.NONE).fit(split.xTrain, split.tTrain)
val tribuoTree =
    trainTribuoTree(none.transform(split.xTrain), split.tTrain.map { it.toString() }, 5, 1.0f) as TreeModel<Label>
tribuoTree.getTopFeatures(-1)